In [ ]:
from destripeClass import Destriper
import numpy as np  
# os.mkdir("temp")
rastertje = np.load("rasters/cell_63_CDI_2174876.npy")
destriper = Destriper(pad_style='constant', save_plot="temp")
destriper.set_angle(155)
destriper.process(rastertje, plot=True)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Example raster
raster = np.random.random((500, 500))

# Start/end points in array indices (row, col)
r1, c1 = 100, 50
r2, c2 = 400, 300

# Number of sample points
num = 500

# Interpolate pixel positions along the line
rows = np.linspace(r1, r2, num)
cols = np.linspace(c1, c2, num)

# Sample raster values using nearest neighbor
values = raster[rows.astype(int), cols.astype(int)]

# Distance (in pixels unless you know pixel size)
dist = np.linspace(0, 1, num)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.imshow(raster, cmap='gray')
plt.plot([c1, c2], [r1, r2], 'r-')  # Line between points
plt.scatter([c1, c2], [r1, r2], c='red')  # Start/end points
plt.title("Raster with Line")
plt.subplot(1, 2, 2)
plt.plot(dist, values)
plt.xlabel("Relative distance")
plt.ylabel("Value")
plt.title("Cross-section from NumPy raster")
plt.grid(True)
# plt.show()


In [ ]:
from difClass import DifferenceRaster, BathymetryRaster
import numpy as np
from destripeClass import Destriper
destriper = Destriper(pad_style='constant')

In [ ]:
raster = np.load("destriped_rasters/cell_72_CDI_2174760_destriped.npy")
bat_raster = BathymetryRaster(raster)
bat_raster.analyse_spectrum()
# bat_raster.plot_raster(bat_raster.raster_demeaned, title="Raster (Demeaned)", show=True)
# bat_raster.plot_raster(bat_raster.magnitude_spectrum, title="Magnitude Spectrum", show=True)

raster2 = destriper._extrapolate_to_square(raster, degree=4, method='nearest_smooth')
bat_raster.plot_raster(bat_raster.raster_demeaned, title="Raster Extrapolated to Square", show=True)

In [ ]:
bat_raster = BathymetryRaster(raster2)
bat_raster.analyse_spectrum()
bat_raster.plot_raster(bat_raster.raster_demeaned, title="Raster Extrapolated to Square (Demeaned)", show=True)
bat_raster.plot_raster(bat_raster.magnitude_spectrum, title="Magnitude Spectrum of Extrapolated Raster", show=True)

magnitude_spectrum = np.fft.fft2(bat_raster.raster_demeaned)
magnitude_spectrum_shifted = np.fft.fftshift(magnitude_spectrum)
bat_raster.plot_raster(np.log(np.abs(magnitude_spectrum_shifted) + 1), title="Log Magnitude Spectrum (Shifted)", show=True)

cutoff = 4
low_pass_filter = np.zeros_like(magnitude_spectrum_shifted)
low_pass_filter[magnitude_spectrum_shifted.shape[0]//2-cutoff:magnitude_spectrum_shifted.shape[0]//2+cutoff,
                magnitude_spectrum_shifted.shape[1]//2-cutoff:magnitude_spectrum_shifted.shape[1]//2+cutoff] = 1

# low_pass_filter = np.ones_like(magnitude_spectrum_shifted)

magnitude_spectrum_filtered = magnitude_spectrum_shifted * low_pass_filter
magnitude_spectrum_filtered_shifted_back = np.fft.ifftshift(magnitude_spectrum_filtered)
raster_filtered = np.fft.ifft2(magnitude_spectrum_filtered_shifted_back).real
bat_raster.plot_raster(raster_filtered, title="Low-pass Filtered Raster", show=True)
bat_raster.plot_raster(bat_raster.raster_demeaned - raster_filtered, title="Difference Between Extrapolated and Filtered Raster", show=True)



In [ ]:
raster3 = bat_raster.raster_demeaned - raster_filtered

def local_std(raster, window_size):
    """Calculate local standard deviation of a raster."""
    from scipy.ndimage import generic_filter
    return generic_filter(raster, np.nanstd, size=window_size)

local_std_raster = local_std(bat_raster.raster_demeaned, window_size=10)
local_std_raster[bat_raster.nan_mask] = np.nan  # Set no-data areas to NaN
bat_raster.plot_raster(local_std_raster, title="Local Standard Deviation (10x10)", show=True)

In [ ]:
from scipy.ndimage import gaussian_filter
def compute_gradient_magnitude(raster):
    """Compute the gradient magnitude of a raster."""
    from scipy.ndimage import sobel
    dx = sobel(raster, axis=1)  # Horizontal gradient
    dy = sobel(raster, axis=0)  # Vertical gradient
    grad = np.hypot(dx, dy)  # Gradient magnitude
    
    # smooth the gradient magnitude to reduce noise
    
    grad = gaussian_filter(grad, sigma=2)
    return grad

grad = compute_gradient_magnitude(raster3)

bat_raster.plot_raster(grad, title="Gradient Magnitude", show=True)

grad_of_grad = compute_gradient_magnitude(grad)
bat_raster.plot_raster(grad_of_grad, title="Gradient of Gradient Magnitude", show=True)


In [ ]:
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
import seaborn as sns
import pandas as pd
from scipy import ndimage

In [ ]:
max1 = ndimage.maximum_filter(raster3, size=15)
min1 = ndimage.minimum_filter(raster3, size=15)
bat_raster.plot_raster(max1, title="Local Maxima (15x15)", show=True)
bat_raster.plot_raster(min1, title="Local Minima (15x15)", show=True)

In [ ]:
feat_1 = local_std_raster.flatten()
feat_2 = grad.flatten()
feat_3 = grad_of_grad.flatten()
feat_4 = np.abs(raster3).flatten()
feat_5 = max1.flatten()
feat_6 = min1.flatten()

clusters  = KMeans(n_clusters=2).fit_predict(np.stack([feat_1, feat_2, feat_3, feat_4, feat_5, feat_6], axis=1))

clusters2d = clusters.reshape(bat_raster.raster_demeaned.shape)
clusters2d[np.isnan(raster)] = -1  # Set no-data areas to a separate cluster
bat_raster.plot_raster(clusters2d, title="KMeans Clusters", show=True)

In [ ]:
notch = destriper._create_notch(destriper.padded_residuals_window)
import matplotlib.pyplot as plt
plt.figure(figsize=(6, 6))    
plt.imshow(-1 * notch, cmap='gray')
plt.title("Notch Filter at 0°")
h, w = notch.shape
plt.xticks([0, w//2, w], labels=[f"-{w//2}", "0", f"{w//2}"])
plt.yticks([0, h//2, h], labels=[f"{h//2}", "0", f"-{h//2}"])

plt.xlabel("Frequency (cycles/pixel)")
plt.ylabel("Frequency (cycles/pixel)")
plt.grid()
plt.savefig("temp/notch_filter.png", dpi=300)

In [1]:
import numpy as np
from difClass import BathymetryRaster

In [4]:
raster = np.load("destriped_rasters/cell_14_CDI_2174768_destriped.npy")
bat_ras = BathymetryRaster(raster)
length, width = bat_ras.raster.shape
bat_ras.cross_section(bat_ras.raster, (length-1, 0), (0, width-1), num=500, plot=True, save_path="temp/cross_section.png");